# 03 Topic Modeling

## Description
Applies BERTopic to the labeled corpus to discover latent topic structures and their evolution over time.
Two analyses are performed:
1. **Topic model** – BERTopic clustering of documents; topic information and representative documents exported
2. **Topic trends** – Monthly topic share by actor (State / Institution), with pagination contamination removed and TF-IDF auto-labels

## Input
- `corpus/analysis_corpus_labeled.csv` – Labeled analysis corpus (output of `01_preprocessing.ipynb`)

## Output
- `outputs/analysis_with_topics_labeled.csv` – Corpus with topic assignments
- `outputs/topic_info_labeled.csv` – Topic summary table
- `outputs/top_docs_by_topic_labeled.csv` – Representative documents per topic
- `outputs/fig_topic_trends_State.png` – Topic share trends (State actors)
- `outputs/fig_topic_trends_Institution.png` – Topic share trends (Institutional actors)
- `outputs/fig_topic_legend_State.png` – Topic structure overview (State)
- `outputs/fig_topic_legend_Institution.png` – Topic structure overview (Institution)


In [ ]:
# Install BERTopic if needed
# !pip install bertopic umap-learn hdbscan sentence-transformers
import os, shutil
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from bertopic import BERTopic

INPUT   = "./corpus/analysis_corpus_labeled.csv"
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
SAMPLE  = 500   # Set to 0 for full corpus
SEED    = 42


In [ ]:
# Load and (optionally) sample corpus
def load_text(p):
    try:
        with open(p, "r", encoding="utf-8") as f:
            t = f.read()
        return (t[:6000] + "\n...\n" + t[-2000:]) if len(t) > 9000 else t
    except Exception: return ""

df = pd.read_csv(INPUT, dtype=str)
df = df[df["clean_path"].notna()].copy()
if SAMPLE and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE} docs")
docs = [load_text(p) for p in df["clean_path"]]
print(f"Fitting BERTopic on {len(docs)} docs ...")


In [ ]:
# Fit BERTopic
topic_model = BERTopic(language="english", calculate_probabilities=False, verbose=True)
topics, _ = topic_model.fit_transform(docs)
df["topic"] = topics

df.to_csv(f"{OUT_DIR}/analysis_with_topics_labeled.csv", index=False)
info = topic_model.get_topic_info()
info.to_csv(f"{OUT_DIR}/topic_info_labeled.csv", index=False)
print(f"Saved topic_info_labeled.csv  topics={len(info)}")

top_docs = []
for t in sorted(info["Topic"].unique()):
    if t == -1: continue
    sub = df[df["topic"] == t]
    for (actor, country), grp in sub.groupby(["actor","country"]):
        for _, r in grp.head(3).iterrows():
            top_docs.append({"topic":t,"actor":actor,"country":country,
                             "title":r.get("title",""),"url":r.get("url",""),"date":r.get("date","")})
pd.DataFrame(top_docs).to_csv(f"{OUT_DIR}/top_docs_by_topic_labeled.csv", index=False)
print("Saved top_docs_by_topic_labeled.csv")


In [ ]:
# Topic trend visualization (topic_trends_labeled.py logic)
EXTRA_STOPS = {"ukraine","georgia","president","office","russia","news","all","events","current",
               "official","web","site","com","en","gov","ua","www","https","http","html","page",
               "press","releases","release","will","also","new","one","two","three","website"}

def build_topic_labels(df_filtered, n=4):
    by_topic = (df_filtered.groupby("topic")["title"]
                .apply(lambda x: " ".join(x.dropna().astype(str)))
                .reset_index().sort_values("topic").reset_index(drop=True))
    vec = TfidfVectorizer(stop_words="english", max_df=0.85, min_df=1, sublinear_tf=True)
    X = vec.fit_transform(by_topic["title"])
    terms = vec.get_feature_names_out()
    labels = {}
    for i, row in by_topic.iterrows():
        t = int(row["topic"])
        scores = X[i].toarray().flatten()
        top_idx = scores.argsort()[::-1]
        kws = [terms[j] for j in top_idx if terms[j].lower() not in EXTRA_STOPS][:n]
        labels[t] = ", ".join(kws) if kws else f"topic {t}"
    return labels

df2 = pd.read_csv(f"{OUT_DIR}/analysis_with_topics_labeled.csv", dtype=str)
df2["topic"] = pd.to_numeric(df2["topic"], errors="coerce")
df2["month"] = df2.get("month", df2.get("date","")).fillna("").astype(str).str.slice(0,7)
df2 = df2[df2["topic"] != -1].dropna(subset=["topic"]).copy()
df2["topic"] = df2["topic"].astype(int)

# Remove pagination/listing pages
is_pagination = (df2["url"].str.contains(r'\?date-from=', na=False) &
                 df2["url"].str.contains(r'[?&]page=\d', na=False, regex=True))
df2 = df2[~is_pagination].copy()
print(f"Removed {is_pagination.sum()} pagination docs")

top_docs_df = pd.read_csv(f"{OUT_DIR}/top_docs_by_topic_labeled.csv")
topic_labels = build_topic_labels(df2, n=4)

for actor in ["State","Institution"]:
    sub = df2[df2["actor"] == actor]
    if sub.empty: continue
    ct    = sub.pivot_table(index="month", columns="topic", values="url", aggfunc="count", fill_value=0)
    share = ct.div(ct.sum(axis=1), axis=0)
    share.to_csv(f"{OUT_DIR}/topic_share_by_month_{actor.lower()}.csv")
    top_topics = ct.sum(axis=0).sort_values(ascending=False).head(10).index
    plot_df = share[top_topics].sort_index()
    plt.figure(figsize=(13,5))
    for t in plot_df.columns:
        plt.plot(plot_df.index, plot_df[t], label=f"T{t}: {topic_labels.get(t,str(t))}")
    plt.xticks(rotation=60, ha="right", fontsize=8)
    plt.title(f"Topic Share over Time (Top 10) — {actor}")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig_topic_trends_{actor}.png", dpi=150); plt.close()
    print(f"Saved fig_topic_trends_{actor}.png")
